#### Messages
Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

Role - Identifies the message type (e.g. system, user)
Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
Metadata - Optional fields such as response information, message IDs, and token usage
LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [15]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")

In [16]:
model.invoke("What is langchain")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user asks "What is langchain". This is a straightforward definition/explanation request about a specific technology/framework.\n\n2.  **Identify Key Subject**: LangChain is a popular open-source framework/library for developing applications powered by large language models (LLMs).\n\n3.  **Core Concepts to Cover**:\n   - What it is (framework/library)\n   - Primary purpose (building LLM-powered applications)\n   - Key features/capabilities (chains, agents, memory, tools, integrations)\n   - Why it\'s useful (abstraction, modularity, ecosystem)\n   - Common use cases (chatbots, RAG, data analysis, automation)\n   - Current status/ecosystem (LangChain, LangGraph, LangSmith, open-source, Python/JS)\n   - Brief mention of alternatives/context (not required but helpful for perspective)\n\n4.  **Structure the Response**:\n   - Definition/Overview\n   - Key Components/Features\n   - Why Developers 

Use text prompts when:

1. You have a single, standalone request
2. You don’t need conversation history
3. You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types

1. System message - Tells the model how to behave and provide context for interactions
2. Human message - Represents user input and interactions with the model
3. AI message - Responses generated by the model, including text content, tool calls, and metadata
4. Tool message - Represents the outputs of tool calls

#### System Message

A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses

#### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

#### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

#### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [22]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage 
from pprint import pprint
messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a small poem on artificial intelligence")
]

response = model.invoke(messages)
pprint(response.content)

('\n'
 '<think>\n'
 "Here's a thinking process:\n"
 '\n'
 '1.  **Analyze User Request:**\n'
 '   - **Topic:** Artificial Intelligence (AI)\n'
 '   - **Format:** Small poem\n'
 '   - **Role:** Poetry expert (implies quality, structure, imagery, and '
 'thematic depth)\n'
 '\n'
 '2.  **Identify Key Themes/Imagery for AI:**\n'
 '   - Machines learning, thinking, mimicking\n'
 '   - Code, circuits, silicon, data\n'
 '   - Human vs. machine, creation vs. creator\n'
 '   - Light, electricity, networks, algorithms\n'
 '   - Paradoxes: cold logic vs. emergent "warmth"/creativity\n'
 '   - Future, evolution, silent observer/participant\n'
 '\n'
 '3.  **Determine Poetic Structure:**\n'
 '   - "Small poem" suggests 3-4 stanzas, maybe 4-6 lines each, or a single '
 'tight stanza.\n'
 "   - I'll aim for 3 stanzas of 4 lines (12 lines total) for balance.\n"
 '   - Rhyme scheme: Maybe ABAB or ABCB for musicality, or free verse with '
 "strong rhythm. I'll go with a loose ABAB/ABCB hybrid for accessib

In [23]:
ai_msg = AIMessage("I'd be happy to help you with that question")

messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
pprint(response.content)

('\n'
 '<think>\n'
 "Here's a thinking process:\n"
 '\n'
 '1.  **Analyze User Input:** The user asks "Great! What\'s 2+2?"\n'
 '2.  **Identify Core Task:** Simple arithmetic question.\n'
 '3.  **Determine Answer:** 2 + 2 = 4.\n'
 '4.  **Formulate Response:** Keep it direct, accurate, and friendly. "2 + 2 '
 'equals 4." or just "4".\n'
 "5.  **Check for Constraints/Context:** None. It's a straightforward math "
 'question.\n'
 '6.  **Final Output Generation:** "2 + 2 equals 4. Let me know if you need '
 'help with anything else!" (Keep it concise and helpful)✅\n'
 '</think>\n'
 '\n'
 '2 + 2 equals **4**. Let me know if you have any other questions!')


In [28]:
human_msg = HumanMessage(
    content="Hello!",
    name = "alice",
    id = "msg_123"
)

response = model.invoke([human_msg])
response
# pprint(response.content)

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hello!"\n   - This is a standard greeting.\n   - No specific question or request is attached.\n\n2.  **Identify Intent:**\n   - The user is initiating a conversation.\n   - Expected response: A friendly greeting back, possibly with an offer to help.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting warmly.\n   - Keep it concise.\n   - Invite the user to share what they need help with.\n   - Maintain a helpful and professional tone.\n\n4.  **Draft Response (Mental):**\n   Hello! How can I assist you today? 😊\n\n5.  **Refine Response:**\n   - Check tone: Friendly, professional, open-ended.\n   - Check length: Short and to the point.\n   - No extra fluff.\n   - Matches standard AI assistant behavior.\n\n   Final: "Hello! How can I help you today?" (or similar)\n\n6.  **Output Generation:** (Proceeds to output)✅\n</think>\n\nHello! How can I help you today? 😊', ad

In [29]:
from langchain.messages import ToolMessage
ai_message = AIMessage(
    content=[],
    tool_calls = [
        {
            "name" : "get_weather",
            "args" : {"location" : "San Francisco"},
            "id" : "call_123"
        }
    ]
)

weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content= weather_result,
    tool_call_id= "call_123"    # must match the call ID
)

messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]

response = model.invoke(messages)

tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [31]:
response

AIMessage(content='\n<think>\nThe user asked for the weather in San Francisco. The tool response indicates it is sunny and 72°F. I will provide this information directly to the user.\n</think>\n\nThe weather in San Francisco is sunny with a temperature of 72°F.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 69, 'total_tokens': 122, 'completion_time': 0.100706048, 'completion_tokens_details': None, 'prompt_time': 0.004712212, 'prompt_tokens_details': None, 'queue_time': 0.047986518, 'total_time': 0.10541826}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fe0d3-d28a-7cf0-ba8a-5358d7bf045e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 53, 'total_tokens': 122})